In [1]:
import pandas as pd
import os, dill
import numpy as np
import cvxpy as cp
from typing import Iterable, Optional, Mapping, cast
from plotly import graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

from ecoli.processes.metabolism_redux_classic import NetworkFlowModel, FlowResult

os.chdir(os.path.expanduser('~/dev/vEcoli'))

%load_ext autoreload

In [2]:
# import sim
time = '600'
date = '2026-01-22'
experiment = 'homeostatic_only'
condition = 'basal'
entry = f'{experiment}_{time}_{date}'
folder = f'out/objective_weight/{condition}/{entry}/'

output = np.load(folder + '0_output.npy',allow_pickle='TRUE').item()
output = output['agents']['0']
fba = output['listeners']['fba_results']
bulk = pd.DataFrame(output['bulk'])
f = open(folder + 'agent_steps.pkl', 'rb')
agent = dill.load(f)
f.close()

metabolism = agent['ecoli-metabolism-redux-classic']

In [3]:
stoichiometry = metabolism.stoichiometry.copy()
reaction_names = metabolism.reaction_names
metabolites = metabolism.metabolite_names.copy()
counts_to_molar = output['listeners']['enzyme_kinetics']['counts_to_molar']

homeostatic_only_term = np.array(fba['homeostatic_term'].copy())/counts_to_molar # covert to counts
homeostatic_only_baseline = np.max(homeostatic_only_term) # in counts

homeostatic_dm_targets =  pd.DataFrame(fba['target_homeostatic_dmdt'], columns=metabolism.homeostatic_metabolites).mul(counts_to_molar, axis=0).max(axis=0)
homeostatic_metabolite_counts = pd.DataFrame(fba['homeostatic_metabolite_counts'], columns=metabolism.homeostatic_metabolites).mul(counts_to_molar, axis=0).mean(axis=0)
maintenance = pd.DataFrame(fba["maintenance_target"][1:], columns=['maintenance_reaction']).mul(counts_to_molar[1:], axis=0).mean(axis=0)
kinetic = pd.DataFrame(fba["target_kinetic_fluxes"], columns=metabolism.kinetic_constraint_reactions).mul(counts_to_molar, axis=0).mean(axis=0)

print(len(homeostatic_dm_targets))
len(homeostatic_metabolite_counts)

172


172

In [4]:
FREE_RXNS = [
    "TRANS-RXN-145",
    "TRANS-RXN0-545",
    "TRANS-RXN0-474",
    "ATPSYN-RXN (reverse)",
]

def test_NetworkFlowModel(weights, solver_choice = cp.GLOP,
                          homeostatic_metabolite_counts = homeostatic_metabolite_counts,
                          homeostatic_dm_targets = homeostatic_dm_targets,
                          kinetic=kinetic,
                          binary_kinetics_idx=None,
                          maintenance=maintenance,
                          ):
    model = NetworkFlowModel(
            stoich_arr=stoichiometry,
            metabolites=metabolites,
            reactions=reaction_names,
            homeostatic_metabolites=metabolism.homeostatic_metabolites,
            kinetic_reactions=metabolism.kinetic_constraint_reactions,
            free_reactions=FREE_RXNS)
    model.set_up_exchanges(exchanges=metabolism.exchange_molecules, uptakes=metabolism.allowed_exchange_uptake)
    solution: FlowResult = model.solve(
            homeostatic_concs=homeostatic_metabolite_counts, # in conc
            homeostatic_dm_targets=np.array(list(dict(homeostatic_dm_targets).values())), # conc
            maintenance_target=maintenance, # conc range
            kinetic_targets=np.array(list(dict(kinetic).values())), # conc range
            binary_kinetic_idx=binary_kinetics_idx,
            # force_flow_idx=force_reaction_idx,
            objective_weights=weights, #same
            upper_flux_bound= 100, # increase to 10^9 because notebook runs FlowResult using Counts if directly imported from listeners, WC runs using conc.
            target_minimal_flux=counts_to_molar[-1],
            solver=solver_choice) #SCS. ECOS, MOSEK
    return solution.objective, solution.homeostatic_term, solution.diversity_term, solution.velocities, solution.dm_dt

In [5]:
def met_homeo(MOI, dmdt):
    ddt = dmdt.loc[MOI]
    return np.abs(ddt-homeostatic_dm_targets.loc[MOI])/homeostatic_metabolite_counts[MOI]

# Pairwise with Efficiency

In [ ]:
lambda_eff_range = np.logspace(-10, -4, 1000)
acceptable_lambda = []
homeo_diff = []
tol = 0.1
pareto = dict()

for lam in lambda_eff_range:
    objective_weights = {
        "efficiency": lam,
        "homeostatic": 1,
    }

    try :
        obj_val, home_term, eff_term, v, dmdt = test_NetworkFlowModel(objective_weights)

        pareto[lam] = dict()
        pareto[lam]['Homeostatic Term'] = home_term
        pareto[lam]['Weighted Efficiency Term'] = eff_term
        pareto[lam]['Unweighted Efficiency Term'] = eff_term/lam

        flux = pd.DataFrame(v, index=reaction_names)
        met_dt = pd.DataFrame(dmdt, index=metabolism.metabolite_names)

        CPD12261_dt = met_dt.loc['CPD-12261[p]']
        glycomono_dt = met_dt.loc["glycogen-monomer[c]"]

        CPD12261_home = met_homeo('CPD-12261[p]', met_dt)
        permitted_home = home_term - met_homeo("glycogen-monomer[c]", met_dt)

        if np.isclose(CPD12261_home, permitted_home):
            acceptable_lambda.append(lam)
    except:
        continue

## Pareto Curve with Efficiency

In [ ]:
pareto_df = (
    pd.DataFrame.from_dict(pareto, orient="index")
      .rename_axis("lambda_eff")
      .reset_index()
      .sort_values("lambda_eff")
)

# Safety: drop any weird values
pareto_df = pareto_df.replace([np.inf, -np.inf], np.nan).dropna()


In [ ]:
import plotly.express as px
import plotly.express.colors as pc

# Pastel palette (qualitative). Take the first two colors as endpoints.
cmin, cmax = pc.qualitative.Pastel[0], pc.qualitative.Pastel[2]

df = pareto_df.copy()
df = df.sort_values("lambda_eff")

# # Axes (typically log makes sense)
# fig.update_xaxes(type="log", title="Weighted Efficiency Term (eff_term / λ)")
#
# # Only log y if strictly > 0
# if (df["Homeostatic Term"] > 0).all():
#     fig.update_yaxes(type="log", title="Homeostatic Term")
# else:
#     fig.update_yaxes(title="Homeostatic Term")

# Make the colorbar log-scaled by transforming the color variable
# (recommended because λ spans orders of magnitude)
df["log10_lambda_eff"] = np.log10(df["lambda_eff"])
fig = px.scatter(
    df,
    x="Weighted Efficiency Term",
    y="Homeostatic Term",
    color="log10_lambda_eff",
    color_continuous_scale=[cmin, cmax],
    hover_data={"lambda_eff": ":.2e"},
    title="Pareto plot: Homeostatic vs Efficiency (colored by log10 λeff)",
)
fig.update_xaxes(type="log")
if (df["Homeostatic Term"] > 0).all():
    fig.update_yaxes(type="log")
fig.update_coloraxes(colorbar_title="log10(λeff)")

# Add pareto curve
best_y = np.inf
best_x = np.inf
frontier_mask = []
for i in range(len(df)):
    y = df.loc[i, "Homeostatic Term"]
    x = df.loc[i, "Weighted Efficiency Term"]
    if y < best_y:
        frontier_mask.append(True)
        best_y = y
    elif x < best_x:
        frontier_mask.append(True)
        best_x = x
    else:
        frontier_mask.append(False)

frontier = df[frontier_mask]

fig.add_trace(go.Scatter(
    x=frontier["Weighted Efficiency Term"],
    y=frontier["Homeostatic Term"],
    mode="lines+markers",
    line=dict(color=pc.qualitative.Pastel[1], dash="dot"),
    showlegend=False
))

# change order of trace
fig.data = (fig.data[1], fig.data[0])

fig.update_layout(
    template="plotly_white",
    # paper_bgcolor='rgba(255, 0, 0, 0)',
    # plot_bgcolor='rgba(255, 0, 0, 0)',
)
fig.show()
# fig.write_image(f'/Users/HeenaSaqib/dev/vEcoli/notebooks/Heena notebooks/Metabolism_New Genes/out/objective_weights/homeostatic_efficiency/'
#                 f'pareto_frontier.png', width=800, height=600, scale=3)

# Pairwise with Kinetics

In [6]:
binary_kinetic_idx = metabolism.binary_kinetic_idx

In [118]:
homeostatic_dm_targets =  pd.DataFrame(fba['target_homeostatic_dmdt'], columns=metabolism.homeostatic_metabolites).mul(counts_to_molar, axis=0).max(axis=0)
homeostatic_metabolite_counts = pd.DataFrame(fba['homeostatic_metabolite_counts'], columns=metabolism.homeostatic_metabolites).mul(counts_to_molar, axis=0).mean(axis=0)
maintenance = pd.DataFrame(fba["maintenance_target"][1:], columns=['maintenance_reaction']).mul(counts_to_molar[1:], axis=0).mean(axis=0)
kinetic = pd.DataFrame(fba["target_kinetic_fluxes"], columns=metabolism.kinetic_constraint_reactions).mul(counts_to_molar, axis=0).mean(axis=0)

n = 1000
lambda_range = np.logspace(-10, 0, n)
acceptable_lambda = []
homeo_diff = []
tol = 1E-7
pareto = dict()
homeo_baseline = 0
count = 0

for lam in lambda_range:
    objective_weights = {
        "kinetics": lam,
        "homeostatic": 1,
    }

    try :
        obj_val, home_term, kin_term, v, dmdt = test_NetworkFlowModel(objective_weights,
                                                                      homeostatic_metabolite_counts = homeostatic_metabolite_counts,
                                                                      homeostatic_dm_targets = homeostatic_dm_targets,
                                                                      kinetic=kinetic,
                                                                      maintenance=maintenance,
                                                                      binary_kinetics_idx=None)
        if lam == lambda_range[0]:
            homeo_baseline = home_term

        pareto[lam] = dict()
        pareto[lam]['Homeostatic Term'] = home_term
        pareto[lam]['Weighted Kinetics Term'] = kin_term
        pareto[lam]['Unweighted Kinetics Term'] = kin_term/lam

        flux = pd.DataFrame(v, index=reaction_names)
        met_dt = pd.DataFrame(dmdt, index=metabolism.metabolite_names)

        CPD12261_dt = met_dt.loc['CPD-12261[p]']
        glycomono_dt = met_dt.loc["glycogen-monomer[c]"]

        CPD12261_home = met_homeo('CPD-12261[p]', met_dt).values
        glycomono_home = met_homeo("glycogen-monomer[c]", met_dt).values
        permitted_home = home_term - glycomono_home

        if ((np.isclose(CPD12261_home, permitted_home) or CPD12261_home < counts_to_molar[1]) and
                np.isclose(home_term, homeo_baseline)):
            count+=1
            acceptable_lambda.append(lam)
    except:
        continue

In [119]:
pareto_df = (
    pd.DataFrame.from_dict(pareto, orient="index")
      .rename_axis("lambda_kin")
      .reset_index()
      .sort_values("lambda_kin")
)

# Safety: drop any weird values
pareto_df = pareto_df.replace([np.inf, -np.inf], np.nan).dropna()


In [196]:
import plotly.express as px
import plotly.express.colors as pc

# Pastel palette (qualitative). Take the first two colors as endpoints.
cmin, cmax = pc.qualitative.Pastel[0], pc.qualitative.Pastel[2]

df = pareto_df.copy()
df = df.sort_values("lambda_kin")

# Make the colorbar log-scaled by transforming the color variable
# (recommended because λ spans orders of magnitude)
df["log10_lambda_kin"] = np.log10(df["lambda_kin"])
fig = px.scatter(
    df,
    x="Weighted Kinetics Term",
    y="Homeostatic Term",
    color="log10_lambda_kin",
    color_continuous_scale=[cmin, cmax],
    hover_data={"lambda_kin": ":.2e"},
    title=f"Pareto plot: Homeostatic vs Kinetics (colored by log10 λkin) Mean all, Max dmdt, Binary=F, n = {n}",
)

fig.update_xaxes(type="log")
if (df["Homeostatic Term"] > 0).all():
    fig.update_yaxes(type="log")
fig.update_coloraxes(colorbar_title="log10(λkin)")
fig.update_xaxes(type="log")

# Add pareto curve
elbow = [np.inf, np.inf]
elbow_idx = 0
for i in range(len(df)):
    y = df.loc[i, "Homeostatic Term"]
    x = df.loc[i, "Weighted Kinetics Term"]

    if x <= elbow[0] and y <= elbow[1]:
        elbow = [x, y]
        elbow_idx = i

best_x = elbow[0]
best_y = elbow[1]

screen_y = np.inf
# split the graph into lowest x and lowest y comparison
# first 0-i is comparing y , i+1:end is comparing x
frontier_mask = [True] * len(df)

for i in np.arange(elbow_idx):
    y = df.loc[i, "Homeostatic Term"]

    if best_y < y < screen_y:
        frontier_mask[i] = True
        screen_y = y
    else:
        frontier_mask[i] = False

screen_x = 0

for i in np.arange(elbow_idx, len(df)):
    x = df.loc[i, "Weighted Kinetics Term"]

    if best_x > x:
        frontier_mask[i] = True
        best_x = x
    else:
        frontier_mask[i] = False

frontier = df.loc[frontier_mask].sort_values("Weighted Kinetics Term")

fig.add_trace(go.Scatter(
    x=frontier["Weighted Kinetics Term"],
    y=frontier["Homeostatic Term"],
    mode="lines+markers",
    line=dict(color=pc.qualitative.Pastel[1], dash="dot"),
    showlegend=False
))

# change order of trace
fig.data = (fig.data[1], fig.data[0])

fig.update_layout(
    template="plotly_white",
    paper_bgcolor='rgba(255, 0, 0, 0)',
    plot_bgcolor='rgba(255, 0, 0, 0)',
)
fig.show(renderer="browser")
# fig.write_image(f'/Users/HeenaSaqib/dev/vEcoli/notebooks/Heena notebooks/Metabolism_New Genes/out/objective_weights/homeostatic_kinetics/'
#                 f'pareto_frontier_max_dmdt.png', width=1200, height=600, scale=3)
# fig.write_html(f'/Users/HeenaSaqib/dev/vEcoli/notebooks/Heena notebooks/Metabolism_New Genes/out/objective_weights/homeostatic_kinetics/'
#                 f'pareto_frontier_max_dmdt.html')

# Pairewise with Secretion

In [85]:
homeostatic_dm_targets =  pd.DataFrame(fba['target_homeostatic_dmdt'], columns=metabolism.homeostatic_metabolites).mul(counts_to_molar, axis=0).max(axis=0)
homeostatic_metabolite_counts = pd.DataFrame(fba['homeostatic_metabolite_counts'], columns=metabolism.homeostatic_metabolites).mul(counts_to_molar, axis=0).mean(axis=0)
maintenance = pd.DataFrame(fba["maintenance_target"][1:], columns=['maintenance_reaction']).mul(counts_to_molar[1:], axis=0).mean(axis=0)
kinetic = pd.DataFrame(fba["target_kinetic_fluxes"], columns=metabolism.kinetic_constraint_reactions).mul(counts_to_molar, axis=0).mean(axis=0)
binary_kinetic_idx = metabolism.binary_kinetic_idx

n = 50
lambda_secretion_range = np.logspace(-10, 0, n)
acceptable_lambda = []
homeo_diff = []
tol = 0.1
pareto = dict()

for lam in lambda_secretion_range:
    objective_weights = {
        "secretion": lam,
        "homeostatic": 1,
    }

    try :
        obj_val, home_term, secretion_term, v, dmdt = test_NetworkFlowModel(objective_weights,
                                                                            binary_kinetics_idx=None)

        pareto[lam] = dict()
        pareto[lam]['Homeostatic Term'] = home_term
        pareto[lam]['Weighted Secretion Term'] = secretion_term
        pareto[lam]['Unweighted Secretion Term'] = secretion_term/lam

        flux = pd.DataFrame(v, index=reaction_names)
        met_dt = pd.DataFrame(dmdt, index=metabolism.metabolite_names)

        CPD12261_dt = met_dt.loc['CPD-12261[p]']
        glycomono_dt = met_dt.loc["glycogen-monomer[c]"]

        CPD12261_home = met_homeo('CPD-12261[p]', met_dt)
        permitted_home = home_term - met_homeo("glycogen-monomer[c]", met_dt)

        if np.isclose(CPD12261_home, permitted_home):
            acceptable_lambda.append(lam)
    except:
        continue

In [86]:
pareto_df = (
    pd.DataFrame.from_dict(pareto, orient="index")
      .rename_axis("lambda_secretion")
      .reset_index()
      .sort_values("lambda_secretion")
)

# Safety: drop any weird values
pareto_df = pareto_df.replace([np.inf, -np.inf], np.nan).dropna()


In [94]:
import plotly.express as px
import plotly.express.colors as pc

# Pastel palette (qualitative). Take the first two colors as endpoints.
cmin, cmax = pc.qualitative.Pastel[0], pc.qualitative.Pastel[2]

df = pareto_df.copy()
df = df.sort_values("lambda_secretion")

# Make the colorbar log-scaled by transforming the color variable
# (recommended because λ spans orders of magnitude)
df["log10_lambda_secretion"] = np.log10(df["lambda_secretion"])
fig = px.scatter(
    df,
    x="Unweighted Secretion Term",
    y="Homeostatic Term",
    color="log10_lambda_secretion",
    color_continuous_scale=[cmin, cmax],
    hover_data={"lambda_secretion": ":.2e"},
    title=f"Pareto plot: Homeostatic vs Secretion (colored by log10 λsec) Mean all, Max dmdt, Binary=F, n = {n}",
)

fig.update_xaxes(type="log")
if (df["Homeostatic Term"] > 0).all():
    fig.update_yaxes(type="log")
fig.update_coloraxes(colorbar_title="log10(λsec)")
fig.update_xaxes(type="log")

## ------------ Calculate pareto curve ------------ ##
elbow = [np.inf, np.inf]
elbow_idx = 0
for i in range(len(df)):
    y = df.loc[i, "Homeostatic Term"]
    x = df.loc[i, "Unweighted Secretion Term"]

    if x <= elbow[0] and y <= elbow[1]:
        elbow = [x, y]
        elbow_idx = i

best_x = elbow[0]
best_y = elbow[1]

screen_y = np.inf
# split the graph into lowest x and lowest y comparison
# first 0-i is comparing y , i+1:end is comparing x
frontier_mask = [True] * len(df)

for i in np.arange(elbow_idx):
    y = df.loc[i, "Homeostatic Term"]

    if best_y < y < screen_y:
        frontier_mask[i] = True
        screen_y = y
    else:
        frontier_mask[i] = False

screen_x = 0

for i in np.arange(elbow_idx, len(df)):
    x = df.loc[i, "Unweighted Secretion Term"]

    if best_x > x:
        frontier_mask[i] = True
        best_x = x
    else:
        frontier_mask[i] = False

frontier = df.loc[frontier_mask].sort_values("Unweighted Secretion Term")

## ------------ Plot pareto curve ------------ ##
fig.add_trace(go.Scatter(
    x=frontier["Unweighted Secretion Term"],
    y=frontier["Homeostatic Term"],
    mode="lines+markers",
    line=dict(color=pc.qualitative.Pastel[1], dash="dot"),
    showlegend=False
))

# change order of trace
fig.data = (fig.data[1], fig.data[0])

fig.update_layout(
    template="plotly_white",
    paper_bgcolor='rgba(255, 0, 0, 0)',
    plot_bgcolor='rgba(255, 0, 0, 0)',
)
# fig.show(renderer="browser")
fig.write_image(f'/Users/HeenaSaqib/dev/vEcoli/notebooks/Heena notebooks/Metabolism_New Genes/out/objective_weights/homeostatic_secretion/'
                f'pareto_frontier_50.png', width=1200, height=600, scale=3)
fig.write_html(f'/Users/HeenaSaqib/dev/vEcoli/notebooks/Heena notebooks/Metabolism_New Genes/out/objective_weights/homeostatic_secretion/'
                f'pareto_frontier_50.html')

# Pairwise with Diversity

In [319]:
n = 30
binary_kinetic_idx = metabolism.binary_kinetic_idx
lambda_div_range = np.logspace(-10, 0, n)
acceptable_lambda = []
homeo_diff = []
tol = 0.1
pareto = dict()

for lam in lambda_div_range:
    objective_weights = {
        "diversity": lam,
        "homeostatic": 1,
    }

    try :
        obj_val, home_term, div_term, v, dmdt = test_NetworkFlowModel(objective_weights,
                                                                      binary_kinetics_idx=None)

        pareto[lam] = dict()
        pareto[lam]['Homeostatic Term'] = home_term
        pareto[lam]['Weighted Diversity Term'] = div_term
        pareto[lam]['Unweighted Diversity Term'] = div_term/lam

        flux = pd.DataFrame(v, index=reaction_names)
        met_dt = pd.DataFrame(dmdt, index=metabolism.metabolite_names)

        CPD12261_dt = met_dt.loc['CPD-12261[p]']
        glycomono_dt = met_dt.loc["glycogen-monomer[c]"]

        CPD12261_home = met_homeo('CPD-12261[p]', met_dt)
        permitted_home = home_term - met_homeo("glycogen-monomer[c]", met_dt)

        if np.isclose(CPD12261_home, permitted_home):
            acceptable_lambda.append(lam)
    except:
        continue

In [313]:
pareto_df = (
    pd.DataFrame.from_dict(pareto, orient="index")
      .rename_axis("lambda_diversity")
      .reset_index()
      .sort_values("lambda_diversity")
)

# Safety: drop any weird values
pareto_df = pareto_df.replace([np.inf, -np.inf], np.nan).dropna()


## better pareto fontier function

In [296]:
# --- Helper functions for Pareto frontier calculation ---
def _get_line_func(p1, p2):
    """Return a linear function in log-x space passing through two [x, y] points."""
    log_x1, log_x2 = np.log10(p1[0]), np.log10(p2[0])
    m = (p2[1] - p1[1]) / (log_x2 - log_x1)
    return lambda x: p1[1] + m * (np.log10(x) - log_x1)


def _find_extreme_point(x_col, y_col, df, what_to_plot, use_max_x: bool):
    """Return [x, y] and index for the leftmost or rightmost Pareto anchor."""
    extreme_x = np.max(x_col) if use_max_x else np.min(x_col)
    candidates_y = y_col[np.where(x_col == extreme_x)[0]]
    best_y = np.max(candidates_y)
    idx = np.where(y_col == best_y)[0][0]
    return [df.loc[idx, what_to_plot["x"]], df.loc[idx, what_to_plot["y"]]], idx


def _find_elbow(df, what_to_plot):
    """Return [x, y] and index of the elbow (bottom-left dominant point)."""
    elbow, elbow_idx = [np.inf, np.inf], 0
    for i in range(len(df)):
        x = df.loc[i, what_to_plot["x"]]
        y = df.loc[i, what_to_plot["y"]]
        if x <= elbow[0] and y <= elbow[1]:
            elbow, elbow_idx = [x, y], i
    return elbow, elbow_idx


def _collect_pareto_candidates(df, what_to_plot, elbow, line_func_1, line_func_2,
                                leftmost_idx, elbow_idx, rightmost_idx,
                                max_y_point, elbow_point, max_x_point):
    """Build initial pareto_line DataFrame with anchor points + below-line candidates."""
    pareto_line = pd.DataFrame({
        "idx":   [leftmost_idx, elbow_idx, rightmost_idx],
        "x":     [max_y_point[0], elbow_point[0], max_x_point[0]],
        "y":     [max_y_point[1], elbow_point[1], max_x_point[1]],
        "label": ["bicep", "elbow", "forearm"]
    })
    for i in range(len(df)):
        x = df.loc[i, what_to_plot["x"]]
        y = df.loc[i, what_to_plot["y"]]
        if x < elbow[0]: # assign to bicep region
            if y < line_func_1(x):
                pareto_line = pd.concat(
                    [pareto_line, pd.DataFrame({"idx": [i], "x": [x], "y": [y], "label": "bicep"})],
                    ignore_index=True
                )
        else: # assign to forearm region
            # from ipdb import set_trace; set_trace()
            if y < line_func_2(x):
                pareto_line = pd.concat(
                    [pareto_line, pd.DataFrame({"idx": [i], "x": [x], "y": [y], "label": "forearm"})],
                    ignore_index=True
                )
    return pareto_line


def _clean_pareto_segment(pareto_line, label, ascending):
    """Remove points violating monotonicity or convexity within a segment."""
    # monotonicity pass (unchanged)
    segment = (
        pareto_line[pareto_line["label"] == label]
        .sort_values("x", ascending=ascending)
        .reset_index()
    )

    if len(segment) > 1:
        comp_y = np.inf
        for i in range(len(segment)):
            if segment.loc[i, "y"] <= comp_y:
                comp_y = segment.loc[i, "y"]
            else:
                pareto_line = pareto_line.drop(segment.loc[i, "index"])

    # convexity pass via slopes in log-x space
    changed = True
    slope_scale = -1 if label == "forearm" else 1  #flip slope sign for forearm since we expect negative slopes coming from the right
    while changed:
        changed = False
        segment = (
            pareto_line[pareto_line["label"] == label]
            .sort_values("x", ascending=ascending)
            .reset_index()
        )
        if len(segment) < 2:
            break

        # compute slopes between consecutive points in log-x space
        slopes = []
        for i in range(len(segment) - 1):
            dx = (np.log10(segment.loc[i+1, "x"]) - np.log10(segment.loc[i, "x"]))*slope_scale
            dy = segment.loc[i+1, "y"] - segment.loc[i, "y"]
            slopes.append(dy / dx)
        # slopes should be negative and increasing (less negative) for convexity
        # from ipdb import set_trace; set_trace()
        for i in range(len(slopes) - 1):
            if (slopes[i] >= slopes[i+1]) and (slopes[i] < 0) and (slopes[i] != np.inf):  # violation: slope not increasing
                pareto_line = pareto_line.drop(segment.loc[i+1, "index"])
                changed = True
                break  # recompute slopes from scratch after each removal

    return pareto_line

In [314]:
# --- Main Function ---
def pareto_line(plot_x: str, plot_y: str):
    pareto_df.sort_values("lambda_diversity")
    what_to_plot = {"x": plot_x, "y": plot_y}

    x_col = pareto_df[what_to_plot["x"]]
    y_col = pareto_df[what_to_plot["y"]]

    # find the leftmost and rightmost points (anchors) and the elbow
    max_y_point, leftmost_idx  = _find_extreme_point(x_col, y_col, pareto_df, what_to_plot, use_max_x=False)
    max_x_point, rightmost_idx = _find_extreme_point(x_col, y_col, pareto_df, what_to_plot, use_max_x=True)
    elbow, elbow_idx           = _find_elbow(pareto_df.copy(), what_to_plot)

    # construct a line for bicep and forearm segments
    line_func_1 = _get_line_func(max_y_point, elbow)
    line_func_2 = _get_line_func(elbow, max_x_point)

    # collect candidates for pareto frontier: anchors + points below the lines
    pareto_df_candidates = _collect_pareto_candidates(
        pareto_df.copy(), what_to_plot, elbow,
        line_func_1, line_func_2,
        leftmost_idx, elbow_idx, rightmost_idx,
        max_y_point, elbow, max_x_point
    )

    # clean candidates to enforce monotonicity within bicep and forearm segments
    pareto_df_clean = _clean_pareto_segment(pareto_df_candidates, label="bicep",   ascending=True)
    pareto_df_clean = _clean_pareto_segment(pareto_df_clean, label="forearm", ascending=False)

    # sort the dataframe for plotting
    before_elbow = pareto_df_clean[pareto_df_clean["label"] == "bicep"].sort_values(["x","y"], ascending=[True,False])
    elbow_row    = pareto_df_clean[pareto_df_clean["label"] == "elbow"]
    after_elbow  = pareto_df_clean[pareto_df_clean["label"] == "forearm"].sort_values(["x","y"], ascending=[True,True])

    return pd.concat([before_elbow, elbow_row, after_elbow], ignore_index=True)

In [303]:
temp = pareto_line("Weighted Diversity Term", "Homeostatic Term")
temp

/var/folders/s9/gn2yly0s7rzgcc2xyvt7nsxm0000gr/T/ipykernel_44294/1220592110.py:93: RuntimeWarning:

divide by zero encountered in scalar divide

/var/folders/s9/gn2yly0s7rzgcc2xyvt7nsxm0000gr/T/ipykernel_44294/1220592110.py:93: RuntimeWarning:

divide by zero encountered in scalar divide

/var/folders/s9/gn2yly0s7rzgcc2xyvt7nsxm0000gr/T/ipykernel_44294/1220592110.py:93: RuntimeWarning:

divide by zero encountered in scalar divide



,idx,x,y,label
0,27,0.003439,1.425211e-11,bicep
1,17,0.003439,1.203684e-11,bicep
2,13,0.003439,5.094541e-12,elbow
3,0,0.008769,6.095611e-12,forearm
4,1,0.008769,9.157771e-12,forearm


In [320]:
# test adding pareto line to the plot function
import plotly.express as px
import plotly.express.colors as pc

# Pastel palette (qualitative). Take the first two colors as endpoints.
cmin, cmax = pc.qualitative.Pastel[0], pc.qualitative.Pastel[2]

df = pareto_df.copy()
df = df.sort_values("lambda_diversity")

# Make the colorbar log-scaled by transforming the color variable
# (recommended because λ spans orders of magnitude)
df["log10_lambda_diversity"] = np.log10(df["lambda_diversity"])
fig = px.scatter(
    df,
    x="Weighted Diversity Term",
    y="Homeostatic Term",
    color="log10_lambda_diversity",
    color_continuous_scale=[cmin, cmax],
    hover_data={"lambda_diversity": ":.2e"},
    title=f"Pareto plot: Homeostatic vs Diversity (colored by log10 λsec) Mean all, Max dmdt, Binary=F, n = {n}",
)

fig.update_xaxes(type="log")
if (df["Homeostatic Term"] > 0).all():
    fig.update_yaxes(type="log")
fig.update_coloraxes(colorbar_title="log10(λdiv)")
fig.update_xaxes(type="log")

## ------------ Plot pareto curve ------------ ##
pareto_frontier = pareto_line("Weighted Diversity Term", "Homeostatic Term")

fig.add_trace(go.Scatter(
    x=pareto_frontier["x"],
    y=pareto_frontier["y"],
    mode="lines+markers",
    line=dict(color=pc.qualitative.Pastel[1], dash="dot"),
    showlegend=False
))

# change order of trace
fig.data = (fig.data[1], fig.data[0])

fig.update_layout(
    template="plotly_white",
    paper_bgcolor='rgba(255, 0, 0, 0)',
    plot_bgcolor='rgba(255, 0, 0, 0)',
)
fig.show(renderer="browser")
# fig.write_image(f'/Users/HeenaSaqib/dev/vEcoli/notebooks/Heena notebooks/Metabolism_New Genes/out/objective_weights/homeostatic_secretion/'
#                 f'pareto_frontier_50.png', width=1200, height=600, scale=3)
# fig.write_html(f'/Users/HeenaSaqib/dev/vEcoli/notebooks/Heena notebooks/Metabolism_New Genes/out/objective_weights/homeostatic_secretion/'
#                 f'pareto_frontier_50.html')


/var/folders/s9/gn2yly0s7rzgcc2xyvt7nsxm0000gr/T/ipykernel_44294/1220592110.py:93: RuntimeWarning:

divide by zero encountered in scalar divide

/var/folders/s9/gn2yly0s7rzgcc2xyvt7nsxm0000gr/T/ipykernel_44294/1220592110.py:93: RuntimeWarning:

divide by zero encountered in scalar divide

/var/folders/s9/gn2yly0s7rzgcc2xyvt7nsxm0000gr/T/ipykernel_44294/1220592110.py:93: RuntimeWarning:

divide by zero encountered in scalar divide



In [316]:
acceptable_lambda

[np.float64(1e-10),
 np.float64(2.2122162910704502e-10),
 np.float64(4.893900918477499e-10),
 np.float64(1.0826367338740564e-09),
 np.float64(2.395026619987491e-09),
 np.float64(5.298316906283702e-09),
 np.float64(1.1721022975334793e-08),
 np.float64(2.592943797404667e-08),
 np.float64(5.736152510448681e-08),
 np.float64(1.2689610031679235e-07),
 np.float64(2.807216203941181e-07),
 np.float64(6.210169418915616e-07),
 np.float64(1.3738237958832638e-06),
 np.float64(3.039195382313201e-06),
 np.float64(6.723357536499335e-06),
 np.float64(1.4873521072935119e-05),
 np.float64(3.290344562312671e-05),
 np.float64(7.27895384398316e-05),
 np.float64(0.00016102620275609426),
 np.float64(0.0003562247890262444),
 np.float64(0.0007880462815669921),
 np.float64(0.0017433288221999908),
 np.float64(0.0038566204211634724),
 np.float64(0.008531678524172814),
 np.float64(0.018873918221350997),
 np.float64(0.04175318936560409),
 np.float64(0.09236708571873885),
 np.float64(0.2043359717856948),
 np.float64

In [70]:
pareto_df.loc[27,"Homeostatic Term"]

np.float64(0.019565709077686108)

In [62]:
pd.DataFrame(dict_pareto)

,idx,x,y
0,8,0.019566,0.019566
1,5,0.004829,0.019566
2,2,0.008783,0.003528


In [61]:
pareto_line

,idx,x,y
0,8,0.019566,0.019566
1,5,0.004829,0.019566
2,2,0.008783,0.003528


In [75]:
# check variations is homeostatic dmdt across _0 and _1
home_dmdt_0 = pd.DataFrame(dmdt_0[metabolism.network_flow_model.homeostatic_idx], index = metabolism.homeostatic_metabolites) * 10000
home_dmdt_1 = pd.DataFrame(dmdt_1[metabolism.network_flow_model.homeostatic_idx], index = metabolism.homeostatic_metabolites) * 10000

not_close_boolean = ~ (home_dmdt_0 == home_dmdt_1)
home_dmdt_not_close_0 = home_dmdt_0[not_close_boolean]
home_dmdt_not_close_1 = home_dmdt_1[not_close_boolean]

In [72]:
home_dmdt_not_close_0

,0
2-3-DIHYDROXYBENZOATE[c],NaN
2-KETOGLUTARATE[c],1.583698
2-PG[c],0.407685
2K-4CH3-PENTANOATE[c],0.611527
4-AMINO-BUTYRATE[c],NaN
...,...
NA+[p],0.454725
OXYGEN-MOLECULE[p],0.536362
FE+3[p],0.747656
CA+2[p],0.568869


In [73]:
home_dmdt_0

,0
2-3-DIHYDROXYBENZOATE[c],0.611527
2-KETOGLUTARATE[c],1.583698
2-PG[c],0.407685
2K-4CH3-PENTANOATE[c],0.611527
4-AMINO-BUTYRATE[c],1.364175
...,...
NA+[p],0.454725
OXYGEN-MOLECULE[p],0.536362
FE+3[p],0.747656
CA+2[p],0.568869


In [17]:
[np.min(acceptable_lambda), np.max(acceptable_lambda)]

[np.float64(1e-10), np.float64(0.001648986944471065)]

In [10]:
[np.min(acceptable_lambda), np.max(acceptable_lambda)]

[np.float64(1e-10), np.float64(0.00121152765862859)]

In [136]:
# def pareto_line(plot_x:str,
#                 plot_y:str
#                 ):
#     pareto_df.sort_values("lambda_diversity")
#     what_to_plot = {
#         "x": plot_x,
#         "y": plot_y
#     }
#
#     x_col = pareto_df[what_to_plot["x"]]
#     y_col = pareto_df[what_to_plot["y"]]
#
#     # find the leftmost point to use for pareto frontier
#     min_x = np.min(x_col)
#     temp_y = y_col[np.where(x_col == min_x)[0]]
#     leftmost_y = np.max(temp_y)
#     leftmost_idx = np.where(y_col==leftmost_y)[0][0]
#
#     # find the rightmost point to use for pareto frontier
#     max_x = np.max(x_col)
#     temp_y = y_col[np.where(x_col == max_x)[0]]
#     rightmost_y = np.max(temp_y)
#     rightmost_idx = np.where(y_col==rightmost_y)[0][0]
#
#     max_y_point = [pareto_df.loc[leftmost_idx, what_to_plot['x']], pareto_df.loc[leftmost_idx, what_to_plot['y']]]
#     max_x_point = [pareto_df.loc[rightmost_idx, what_to_plot['x']], pareto_df.loc[rightmost_idx, what_to_plot['y']]]
#
#     # find the elbow point to use for pareto frontier
#     df = pareto_df.copy()
#     elbow = [np.inf, np.inf]
#     elbow_idx = 0
#     for i in range(len(df)):
#         y = df.loc[i, what_to_plot["y"]]
#         x = df.loc[i, what_to_plot["x"]]
#
#         if x <= elbow[0] and y <= elbow[1]:
#             elbow = [x, y]
#             elbow_idx = i
#
#     # make a function of a line from first to elbow and elbow to last point
#     m1 = (elbow[1] - max_y_point[1])/(elbow[0] - max_y_point[0]) # slope from homeostatic to elbow
#     m2 = (max_x_point[1] - elbow[1])/(max_x_point[0] - elbow[0]) # slope from elbow to pairewise_term x
#     line_func_1 = lambda x: max_y_point[1] + (m1 * (x - max_y_point[0])) #homeostatic to elbow
#     line_func_2 = lambda x: elbow[1] + (m2 * (x - elbow[0])) #elbow to pairwise_term
#
#     # store pareto dataframe
#     dict_pareto = {
#         "idx":[leftmost_idx, elbow_idx, rightmost_idx],
#         "x":[max_y_point[0], elbow[0], max_x_point[0]],
#         "y":[max_y_point[1], elbow[1], max_x_point[1]],
#         "label": ["bicep", "elbow", "forearm"]
#     }
#
#     pareto_line = pd.DataFrame(dict_pareto)
#
#     # start find more pareto points by comparing to line and finding points that are below the line
#     for i in range(len(df)):
#         y = df.loc[i, what_to_plot["y"]]
#         x = df.loc[i, what_to_plot["x"]]
#         if x < elbow[0]: # compare to line_func_1
#             if y < line_func_1(x):
#                 pareto_line = pd.concat([pareto_line, pd.DataFrame({"idx":[i], "x":[x], "y":[y], "label":"bicep"})], ignore_index=True)
#         else: # compare to line_func_2
#             if y < line_func_2(x):
#                 pareto_line = pd.concat([pareto_line, pd.DataFrame({"idx":[i], "x":[x], "y":[y], "label":"forearm"})], ignore_index=True)
#
#     # clean up pareto line dateframe -- remove non-valid points
#     # left side + elbow
#     bicep = pareto_line[pareto_line["label"] == "bicep"].sort_values("x", ascending=True) #key is ascending = T
#     bicep = bicep.reset_index()
#
#     # right side + elbow
#     forearm = pareto_line[pareto_line["label"] == "forearm"].sort_values("x", ascending=False) #key is ascending = F
#     forearm = forearm.reset_index()
#
#     # in bicep, points should have decreasing y from endpoint to elbow
#     if len(bicep) > 1:
#         for i in range(len(bicep)-1):
#             if bicep.loc[i, "y"] <= bicep.loc[i+1, "y"]:
#                 pareto_line = pareto_line.drop(bicep.loc[i+1, "index"])
#
#     # in forearm, points should have decreasing y from endpoint to elbow
#     if len(forearm) > 1:
#         for i in range(len(forearm)-1):
#             if forearm.loc[i, "y"] <= forearm.loc[i+1, "y"]:
#                 pareto_line = pareto_line.drop(forearm.loc[i+1, "index"])
#
#
#     # sort the dataframe for plotting
#     # Split into before / elbow / after using current row order
#     before_elbow = pareto_line.iloc[:1].sort_values("y", ascending=False)
#     elbow_row = pareto_line.iloc[[1]]   # keep as DataFrame
#     after_elbow = pareto_line.iloc[1 + 1:].sort_values("idx", ascending=False)
#
#     # Rejoin
#     pareto_line_sorted = pd.concat(
#         [before_elbow, elbow_row, after_elbow],
#         ignore_index=True
#     )
#
#     return pareto_line_sorted

In [11]:
acceptable_lambda

[np.float64(1e-10),
 np.float64(2.6101572156825333e-10),
 np.float64(6.812920690579622e-10),
 np.float64(1.7782794100389228e-09),
 np.float64(4.641588833612773e-09),
 np.float64(1.2115276586285901e-08),
 np.float64(3.162277660168379e-08),
 np.float64(8.25404185268019e-08),
 np.float64(2.1544346900318867e-07),
 np.float64(5.62341325190349e-07),
 np.float64(1.4677992676220705e-06),
 np.float64(3.8311868495572925e-06),
 np.float64(1e-05),
 np.float64(2.6101572156825386e-05),
 np.float64(6.812920690579622e-05),
 np.float64(0.00017782794100389227),
 np.float64(0.0004641588833612782),
 np.float64(0.00121152765862859)]